# Uso básico da ferramenta Browser com Browser-Use SDK

## Visão Geral

Neste tutorial, aprenderemos como usar o SDK de código aberto Browser-Use com a ferramenta Browser do Amazon Bedrock Agentcore. Forneceremos exemplos de uso da ferramenta de navegador sem interface (headless) e com visualização ao vivo.


### Detalhes do Tutorial


| Informação          | Detalhes                                                                                   |
|:--------------------|:-------------------------------------------------------------------------------------------|
| Tipo de tutorial    | Conversacional                                                                             |
| Tipo de agente      | Único                                                                                      |
| Framework agêntico  | Browser-Use                                                                                |
| Modelo LLM          | Anthropic Claude 3.7 Sonnet                                                                |
| Componentes         | Usando SDK Browser-Use para interagir com a ferramenta de navegador Bedrock Agentcore de forma headless |
| Vertical do tutorial| Cross-vertical                                                                             |
| Complexidade        | Fácil                                                                                      |
| SDK utilizado       | Amazon BedrockAgentCore Python SDK, Browser-Use                                            |

### Arquitetura do Tutorial

Neste tutorial, descreveremos como usar o SDK Browser-Use com a ferramenta de navegador Agentcore.  

Em nosso exemplo, enviaremos instruções em linguagem natural para o agente Browser-Use executar tarefas no navegador Bedrock Agentcore de forma headless.


### Principais Recursos do Tutorial

* Usando a ferramenta de navegador de forma headless
* Usando Browser-Use com a ferramenta de navegador

## Pré-requisitos

Para executar este tutorial, você precisará de:
* Python 3.11+
* Credenciais AWS
* Amazon Bedrock AgentCore SDK
* Browser-Use SDK

## Como funciona

Um sandbox de ferramenta de navegador é um ambiente de execução seguro que permite que agentes IA interajam com segurança com navegadores web. Quando um usuário faz uma solicitação, o Large Language Model (LLM) seleciona as ferramentas apropriadas e traduz os comandos. Esses comandos são executados dentro de um ambiente sandbox controlado contendo um navegador headless e servidor de biblioteca hospedado (usando ferramentas como Playwright). O sandbox fornece isolamento e segurança ao conter interações web dentro de um espaço restrito, prevenindo acesso não autorizado ao sistema. O agente recebe feedback através de capturas de tela e pode realizar tarefas automatizadas enquanto mantém a segurança do sistema. 

![architecture local](../images/browser-tool.png)

## 1. Configurando o Ambiente

#### Por favor, execute o script abaixo para corrigir o problema de headers de autorização e torná-lo compatível com o Amazon Bedrock AgentCore Browser

### 1.1 Instalar as Bibliotecas Pré-Requisito

Primeiro, vamos instalar as bibliotecas necessárias para o cliente sandbox da ferramenta de navegador.

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

### 1.2 Script de Patch para browser-use

In [ ]:
%%writefile patch_browser_use.py

#!/usr/bin/env python3
"""Detectar automaticamente e aplicar patch no session.py do browser_use"""

import os
import shutil
import sys
from pathlib import Path

def find_browser_use_path():
    """Encontrar automaticamente o caminho de instalação do browser_use"""
    try:
        import browser_use
        browser_use_path = Path(browser_use.__file__).parent
        session_file = browser_use_path / "browser" / "session.py"
        return str(session_file)
    except ImportError:
        print("❌ browser_use não instalado. Instale com: pip install browser-use")
        return None

def patch_browser_use():
    # Detectar automaticamente o caminho do arquivo
    file_path = find_browser_use_path()
    if not file_path:
        return False
    
    if not os.path.exists(file_path):
        print(f"❌ Arquivo não encontrado: {file_path}")
        return False
    
    print(f"📁 browser_use encontrado em: {file_path}")
    
    # Criar backup
    backup_path = file_path + ".backup"
    if not os.path.exists(backup_path):
        shutil.copy2(file_path, backup_path)
        print(f"💾 Backup criado: {backup_path}")
    else:
        print(f"📋 Backup já existe: {backup_path}")
    
    # Ler arquivo
    with open(file_path, 'r') as f:
        content = f.read()
    
    # Substituição 1: Adicionar verificação de headers após verificação de cdp_url
    old1 = "if not cdp_url:\n\t\t\tprofile_kwargs['is_local'] = True"
    new1 = "if not cdp_url:\n\t\t\tprofile_kwargs['is_local'] = True\n\n\t\tif headers:\n\t\t\tprofile_kwargs['headers'] = headers"
    
    if old1 in content and "if headers:\n\t\t\tprofile_kwargs['headers'] = headers" not in content:
        content = content.replace(old1, new1)
        print("✅ Verificação de headers adicionada")
    elif "if headers:\n\t\t\tprofile_kwargs['headers'] = headers" in content:
        print("✅ Verificação de headers já existe")
    else:
        print("⚠️ Padrão de verificação de headers não encontrado")
    
    # Substituição 2: Adicionar headers ao CDPClient
    old2 = "self._cdp_client_root = CDPClient(self.cdp_url)"
    new2 = "self._cdp_client_root = CDPClient(self.cdp_url,  additional_headers=self.browser_profile.headers)"
    
    if old2 in content:
        content = content.replace(old2, new2)
        print("✅ Headers adicionados ao CDPClient")
    elif "additional_headers=self.browser_profile.headers" in content:
        print("✅ Headers do CDPClient já existem")
    else:
        print("⚠️ Padrão CDPClient não encontrado")
    
    # Escrever de volta
    with open(file_path, 'w') as f:
        f.write(content)
    
    print("🎉 Patch concluído!")
    return True

if __name__ == "__main__":
    success = patch_browser_use()
    sys.exit(0 if success else 1)

In [ ]:
# Executar o script Python para aplicar o patch para browser-use
!python patch_browser_use.py

In [ ]:
# Reiniciar o Kernel 
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

### 1.3 Importações

Importar as bibliotecas necessárias para inicializar o cliente sandbox da ferramenta de navegador.

In [ ]:
from bedrock_agentcore.tools.browser_client import BrowserClient
from browser_use.llm import ChatAnthropicBedrock, ChatAWSBedrock
from browser_use import Agent
from browser_use import Browser, BrowserProfile
from rich.console import Console
from contextlib import suppress
import asyncio

In [ ]:
console = Console()

## 2. Configurar o cliente do navegador
Vamos configurar o cliente do navegador e aguardar que ele esteja pronto. Em seguida, geraremos a URL do web-socket e headers
 


In [ ]:
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

client = BrowserClient(region)
client.start()

# Extrair ws_url e headers
ws_url, headers = client.generate_ws_headers()

## 4. Função auxiliar para executar tarefa do navegador
Executar uma tarefa de automação de navegador usando o Agente browser-use 




In [ ]:
async def run_browser_task(browser_session: Browser, bedrock_chat: ChatAnthropicBedrock, task: str) -> None:
    """
    Executar uma tarefa de automação de navegador usando browser_use
    
    Args:
        browser_session: Sessão de navegador existente para reutilizar
        bedrock_chat: Instância do modelo de chat Bedrock
        task: Tarefa em linguagem natural para o agente
    """
    try:
        # Mostrar execução da tarefa
        console.print(f"\n[bold blue]🤖 Executando tarefa:[/bold blue] {task}")
        
        # Criar e executar o agente
        agent = Agent(
            task=task,
            llm=bedrock_chat,
            browser_session=browser_session
        )
        
        # Executar com indicador de progresso
        with console.status("[bold green]Executando automação do navegador...[/bold green]", spinner="dots"):
            await agent.run()
        
        console.print("[bold green]✅ Tarefa concluída com sucesso![/bold green]")
        
    except Exception as e:
        console.print(f"[bold red]❌ Erro durante execução da tarefa:[/bold red] {str(e)}")
        import traceback
        if console.is_terminal:
            traceback.print_exc()

## 5. Invocar a tarefa do navegador usando o perfil Browser-use
Criar uma sessão de navegador persistente usando conexão CDP WebSocket e inicializar o modelo Claude do Bedrock para tarefas web automatizadas. Gerenciar o ciclo de vida da sessão com limpeza adequada e executar tarefas de automação do navegador via comandos conduzidos por IA.

In [ ]:
# Criar sessão de navegador e modelo persistentes
browser_session = None
bedrock_chat = None

try:
     # Criar perfil de navegador com headers e timeout 
    browser_profile = BrowserProfile(
        headers=headers,
        timeout=1500000,  # 150 segundos de timeout
    )

    # Criar uma sessão de navegador com CDP URL e keep_alive=True para persistência
    browser_session = Browser(
        cdp_url=ws_url,
        browser_profile=browser_profile,
        keep_alive=True
    )
    
    # Inicializar a sessão do navegador
    console.print("[cyan]🔄 Inicializando sessão do navegador...[/cyan]")
    await browser_session.start()
    
    # Criar ChatBedrockConverse uma vez
    bedrock_chat = ChatAnthropicBedrock(
		model='global.anthropic.claude-haiku-4-5-20251001-v1:0',
		aws_region='us-west-2'
	)
    
    console.print("[green]✅ Sessão do navegador inicializada e pronta para tarefas[/green]\n")

    task = "Procure por uma cafeteira em amazon.com e extraia detalhes da primeira" ## Modifique a tarefa para executar outras tarefas

    await run_browser_task(browser_session, bedrock_chat, task)

finally:
    # Fechar a sessão do navegador
    if browser_session:
        console.print("\n[yellow]🔌 Fechando sessão do navegador...[/yellow]")
        with suppress(Exception):
            await browser_session.close()
        console.print("[green]✅ Sessão do navegador fechada[/green]")


## 6. Limpeza
Parar a sessão do navegador se ainda não foi feito

In [ ]:
client.stop()
print("Sessão do navegador parada com sucesso!")